In [37]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
from pathlib import Path
from sklearn.model_selection import train_test_split,cross_val_score,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,OrdinalEncoder,PowerTransformer
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [5]:
root_dir = Path.cwd().parent
data_dir = root_dir / 'data' / 'interim' / 'urbaneats-cleaned-dataset.csv'

In [6]:
df = pd.read_csv(data_dir)

In [7]:
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,city,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,pickup_time_minutes,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,2,snack,motorcycle,0.0,no,urban,24,INDO,19,3,Saturday,1,11.0,15.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,BANG,25,3,Friday,0,19.0,5.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,BANG,19,3,Saturday,1,8.0,15.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,COIMB,5,4,Tuesday,0,18.0,10.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,CHEN,26,3,Saturday,1,13.0,15.0,afternoon,6.210138,medium


In [8]:
df.shape

(45502, 27)

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:

# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

df
     

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [11]:
# check for missing values

df.isna().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [12]:
dagshub.init(repo_owner='AvanindraBose', repo_name='Urban-Eats-Food-Delivery-Time-Prediction', mlflow=True)

Accessing as AvanindraBose

Initialized MLflow to track repo "AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction"

Repository AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction initialized!

In [13]:
mlflow.set_tracking_uri('https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow')

# Droping Missing Values and then Slecting the 2 Best Model.

In [14]:
temp_df = df.copy().dropna()

In [15]:
temp_df.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [16]:
temp_df.shape

(37695, 16)

In [17]:
X = temp_df.drop(columns= ['time_taken'])
y = temp_df['time_taken']

In [18]:
X.sample(10)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
25273,36.0,4.5,fog,low,1,buffet,motorcycle,0.0,no,metropolitian,0,10.0,night,10.587816,long
18122,29.0,4.3,fog,medium,1,meal,motorcycle,1.0,no,metropolitian,0,10.0,evening,19.761110,very_long
29845,31.0,4.8,sunny,low,0,buffet,motorcycle,0.0,no,metropolitian,1,5.0,night,14.021757,long
24140,27.0,4.7,sunny,low,2,meal,electric_scooter,1.0,no,urban,0,5.0,morning,3.018827,short
28849,31.0,4.5,cloudy,jam,0,buffet,motorcycle,1.0,no,urban,1,5.0,evening,20.179033,very_long
14187,33.0,4.2,windy,jam,0,snack,motorcycle,1.0,no,metropolitian,1,10.0,night,12.236791,long
35953,37.0,4.6,windy,medium,1,buffet,motorcycle,1.0,no,metropolitian,1,15.0,evening,13.973183,long
43133,36.0,4.7,fog,low,0,snack,motorcycle,1.0,no,urban,0,5.0,night,7.790439,medium
21182,23.0,4.9,fog,medium,2,snack,motorcycle,1.0,no,metropolitian,0,15.0,afternoon,5.968814,medium
12217,39.0,4.5,sandstorms,medium,2,buffet,scooter,1.0,no,metropolitian,0,10.0,afternoon,16.722119,very_long


In [19]:
y.sample(10)

19174    23
15566    34
19477    42
10214    14
18240    14
8928     30
30673    27
44053    26
10169    22
17687    25
Name: time_taken, dtype: int64

In [20]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [21]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [22]:
temp_df.dtypes

age                    float64
ratings                float64
weather                 object
traffic                 object
vehicle_condition        int64
type_of_order           object
type_of_vehicle         object
multiple_deliveries    float64
festival                object
city_type               object
time_taken               int64
is_weekend               int64
pickup_time_minutes    float64
order_time_of_day       object
distance               float64
distance_type           object
dtype: object

In [23]:
num_cols = X_train.select_dtypes(include=np.number).columns.to_list()

In [24]:
num_cols.remove('vehicle_condition')
num_cols.remove('multiple_deliveries')

In [25]:
num_cols

['age', 'ratings', 'is_weekend', 'pickup_time_minutes', 'distance']

In [26]:
X_train.select_dtypes(include=object).columns.to_list()

['weather',
 'traffic',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'order_time_of_day',
 'distance_type']

In [27]:
ordinal_cat_cols = ['traffic','distance_type']

nominal_cat_cols = [
    'weather',
    'type_of_order',
    'type_of_vehicle',
    'festival',
    'city_type',
    'order_time_of_day'
]

In [28]:
len(num_cols + nominal_cat_cols + ordinal_cat_cols)

13

In [29]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
)

In [42]:
pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ])

model_pipe_tt = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

In [38]:
scores = cross_validate(
            model_pipe_tt,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

In [39]:
scores

{'fit_time': array([20.5118382 , 20.51391029, 20.39933133, 20.6051693 , 20.60942173]),
 'score_time': array([0.33383083, 0.32973909, 0.37485766, 0.31027222, 0.30884981]),
 'test_mae': array([-3.13546104, -3.12235661, -3.12444186, -3.11345066, -3.12149127]),
 'train_mae': array([-1.15979466, -1.16200418, -1.16248186, -1.16374641, -1.15388512]),
 'test_r2': array([0.82461643, 0.8266513 , 0.82691264, 0.82564512, 0.82780124]),
 'train_r2': array([0.97533032, 0.97527773, 0.97529923, 0.97521568, 0.97559166])}

In [44]:
model_pipe_tt.regressor.named_steps['model'].__class__.__name__

'RandomForestRegressor'

In [40]:
def build_model(trial):
    model_name = trial.suggest_categorical("model", ["RF", "GB", "XGB", "KNN"])

    if model_name == "RF":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("rf_n_estimators", 100, 500),
            max_depth=trial.suggest_int("rf_max_depth", 3, 30),
            min_samples_leaf=trial.suggest_int("rf_min_samples_leaf", 1, 10),
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "GB":
        model = GradientBoostingRegressor(
            n_estimators=trial.suggest_int("gb_n_estimators", 100, 500),
            learning_rate=trial.suggest_float("gb_learning_rate", 0.01, 0.2, log=True),
            max_depth=trial.suggest_int("gb_max_depth", 2, 8),
            random_state=42
        )

    elif model_name == "XGB":
        model = XGBRegressor(
            n_estimators=trial.suggest_int("xgb_n_estimators", 100, 600),
            learning_rate=trial.suggest_float("xgb_learning_rate", 0.01, 0.2, log=True),
            max_depth=trial.suggest_int("xgb_max_depth", 2, 10),
            subsample=trial.suggest_float("xgb_subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("xgb_colsample_bytree", 0.6, 1.0),
            random_state=42,
            n_jobs=-1
        )

    else:
        model = KNeighborsRegressor(
            n_neighbors=trial.suggest_int("knn_n_neighbors", 3, 40),
            weights=trial.suggest_categorical("knn_weights", ["uniform", "distance"])
        )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    model_pipe = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

    return model_pipe

In [45]:
def objective(trial):
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):

        model_pipe = build_model(trial)

        scores = cross_validate(
            model_pipe,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

        train_mae = -scores["train_mae"].mean()
        val_mae = -scores["test_mae"].mean()
        val_mae_std = scores["test_mae"].std()
        train_r2 = scores["train_r2"].mean()
        val_r2 = scores["test_r2"].mean()

        mlflow.log_param("model_type",model_pipe.regressor.named_steps['model'].__class__.__name__)
        mlflow.log_param("trial_number", trial.number)
        mlflow.log_params(trial.params)

        mlflow.log_metric("train_mae_mean", train_mae)
        mlflow.log_metric("val_mae_mean", val_mae)
        mlflow.log_metric("val_mae_std", val_mae_std)
        mlflow.log_metric("train_r2_mean", train_r2)
        mlflow.log_metric("val_r2_mean", val_r2)

        for i, score in enumerate(scores["test_mae"]):
            mlflow.log_metric(f"fold_{i}_val_mae", -score)

        for i, score in enumerate(scores["test_r2"]):
            mlflow.log_metric(f"fold_{i}_val_r2", score)

        trial.set_user_attr("val_mae", val_mae)
        trial.set_user_attr("val_r2", val_r2)

        return val_mae


In [46]:
mlflow.set_experiment('Exp2 : Selecting 2 Best Models')
study = optuna.create_study(direction="minimize",study_name="model_selection")


with mlflow.start_run(run_name= 'Best Model') as parent_run:
    mlflow.log_param("n_trials",50)
    mlflow.log_param("cv_folds",5)
    mlflow.log_param("objective_metric","val_mae")

    study.optimize(objective,n_trials=50)

    best_trial = study.best_trial

    mlflow.log_param("best_trial_number", best_trial.number)
    mlflow.log_param("best_model", best_trial.params["model"])

    for key,value in best_trial.params.items():
        mlflow.log_param(f"best_{key}",value)
    
    mlflow.log_metric("best_cv_mae",best_trial.value)

    trials_df = study.trials_dataframe()
    trials_df.to_csv("optuna_trials.csv", index=False)
    mlflow.log_artifact("optuna_trials.csv")

    best_model_pipe = build_model(best_trial)
    best_model_pipe.fit(X_train,y_train)

    y_pred_train = best_model_pipe.predict(X_train)
    y_pred_test = best_model_pipe.predict(X_test)
    
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.log_params(best_trial.params)

    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)

    mlflow.sklearn.log_model(best_model_pipe, "model")


2026/06/03 06:49:57 INFO mlflow.tracking.fluent: Experiment with name 'Exp2 : Selecting 2 Best Models' does not exist. Creating a new experiment.
[I 2026-06-03 06:49:58,412] A new study created in memory with name: model_selection


🏃 View run trial_0 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2/runs/64fb514aac4d4bac9a151389d1a0d735
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2


[I 2026-06-03 06:50:25,485] Trial 0 finished with value: 3.0933066286260607 and parameters: {'model': 'RF', 'rf_n_estimators': 206, 'rf_max_depth': 30, 'rf_min_samples_leaf': 5}. Best is trial 0 with value: 3.0933066286260607.


🏃 View run trial_1 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2/runs/76f016b5d42c4d05b2817899ac95aa2c
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2


[I 2026-06-03 06:51:03,687] Trial 1 finished with value: 4.449116185038251 and parameters: {'model': 'KNN', 'knn_n_neighbors': 39, 'knn_weights': 'distance'}. Best is trial 0 with value: 3.0933066286260607.


...

🏃 View run trial_48 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2/runs/9e8e1cec89cf4f5dbcad0411e7cb7e05
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2


[I 2026-06-03 07:26:25,913] Trial 48 finished with value: 3.323288433925549 and parameters: {'model': 'RF', 'rf_n_estimators': 427, 'rf_max_depth': 10, 'rf_min_samples_leaf': 5}. Best is trial 42 with value: 3.082608422650648.


🏃 View run trial_49 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2/runs/359b601d919b482f9d2a7582a8e93aeb
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2


[I 2026-06-03 07:27:18,246] Trial 49 finished with value: 3.0880192545134006 and parameters: {'model': 'RF', 'rf_n_estimators': 474, 'rf_max_depth': 15, 'rf_min_samples_leaf': 4}. Best is trial 42 with value: 3.082608422650648.
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
2026/06/03 07:27:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 07:27:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' forma

🏃 View run Best Model at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2/runs/1ad5a330bb7e42099f3b03f3909ab3ad
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/2


In [50]:
exp_data = study.trials_dataframe()

In [55]:
exp_data['params_model'].value_counts()

params_model
RF     32
GB      7
KNN     6
XGB     5
Name: count, dtype: int64

In [63]:
exp_data.groupby(exp_data['params_model'])['value'].agg('mean').sort_values(ascending=True)

params_model
RF     3.202053
XGB    3.353468
GB     3.508924
KNN    4.429659
Name: value, dtype: float64

In [64]:
optuna.visualization.plot_optimization_history(study)

In [65]:
# partial coord plot

optuna.visualization.plot_parallel_coordinate(study,params=["model"])

# From this Experimentation I can conclude that I will be using Random Forest and XgBoost as my underlying models in Stacking Ensemble Technique.